# 准备数据

In [1]:
# 加载模块
from datetime import datetime

from tqdm import tqdm
from xtquant import xtdata

from vnpy.trader.database import DB_TZ

from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import HistoryRequest

from vnpy.alpha import AlphaLab, logger
from logging import INFO
from vnpy.trader.setting import SETTINGS
# SETTINGS["log.active"] = True
# SETTINGS["log.level"] = INFO
# SETTINGS["log.file"] = False
# SETTINGS["log.console"] = True

SETTINGS["database.name"] = "postgresql"
SETTINGS["database.host"] = "localhost"
SETTINGS["database.port"] = "5432"
SETTINGS["database.database"] = "vnpy"
SETTINGS["database.user"] = "guojiantao"
SETTINGS["database.password"] = "123456"
# 配置数据服务
SETTINGS["datafeed.name"] = "xt"
SETTINGS["datafeed.username"] = "client"
SETTINGS["datafeed.password"] = ""

from vnpy.trader.datafeed import get_datafeed


C:\Users\jacke\anaconda3\envs\vnpy_live\Lib\site-packages\xtquant\__init__.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution


xtquant文档地址：http://dict.thinktrader.net/nativeApi/start_now.html


In [2]:
# 设置下载参数
task_name = "csi300"
index_symbol = "000300.SSE"
xt_index_symbol = "000300.SH"

start_date = "20240101"
end_date = "20260415"

intervals = [
    Interval.DAILY,
]

In [3]:
# 创建投研实验室
lab = AlphaLab(f"./lab/{task_name}")    # 指定数据文件夹

In [4]:
# 初始化数据服务（这里配置使用的迅投研）
from vnpy_xt.xt_datafeed import XtDatafeed

datafeed = XtDatafeed()
print(f"datafeed type: {type(datafeed)}")
result = datafeed.init()
print(f"init result: {result}")

datafeed type: <class 'vnpy_xt.xt_datafeed.XtDatafeed'>
init result: True


In [5]:
# 下载历史成分股信息

sector_list = xtdata.get_sector_list()
print(sector_list)
sector_name = '沪深A股'
code_list = xtdata.get_stock_list_in_sector(sector_name)
print(f'{sector_name}一共有{len(code_list)}只品种')
print('-------')
print(code_list)


['上期所', '上证A股', '上证B股', '上证期权', '上证转债', '中金所', '京市A股', '创业板', '大商所', '沪市ETF', '沪市债券', '沪市基金', '沪市指数', '沪深A股', '沪深B股', '沪深ETF', '沪深京A股', '沪深债券', '沪深基金', '沪深指数', '沪深转债', '深市ETF', '深市债券', '深市基金', '深市指数', '深证A股', '深证B股', '深证期权', '深证转债', '科创板', '科创板CDR', '连续合约', '郑商所', '香港联交所指数', '香港联交所股票', 'SW1交通运输', 'SW1交通运输加权', 'SW1休闲服务', 'SW1休闲服务加权', 'SW1传媒', 'SW1传媒加权', 'SW1公用事业', 'SW1公用事业加权', 'SW1农林牧渔', 'SW1农林牧渔加权', 'SW1化工', 'SW1化工加权', 'SW1医药生物', 'SW1医药生物加权', 'SW1商业贸易', 'SW1商业贸易加权', 'SW1国防军工', 'SW1国防军工加权', 'SW1家用电器', 'SW1家用电器加权', 'SW1建筑材料', 'SW1建筑材料加权', 'SW1建筑装饰', 'SW1建筑装饰加权', 'SW1房地产', 'SW1房地产加权', 'SW1有色金属', 'SW1有色金属加权', 'SW1机械设备', 'SW1机械设备加权', 'SW1汽车', 'SW1汽车加权', 'SW1电子', 'SW1电子加权', 'SW1电气设备', 'SW1电气设备加权', 'SW1纺织服装', 'SW1纺织服装加权', 'SW1综合', 'SW1综合加权', 'SW1计算机', 'SW1计算机加权', 'SW1轻工制造', 'SW1轻工制造加权', 'SW1通信', 'SW1通信加权', 'SW1采掘', 'SW1采掘加权', 'SW1钢铁', 'SW1钢铁加权', 'SW1银行', 'SW1银行加权', 'SW1非银金融', 'SW1非银金融加权', 'SW1食品饮料', 'SW1食品饮料加权', 'SW2一般零售', 'SW2一般零售加权', 'SW2专业工程', 'SW2专业工程加权', 'SW2专业零售', 'SW2专业零售加权', 'SW2专用设备'

In [6]:
# 查询交易日历
days = xtdata.get_trading_dates(market="SZ", start_time=start_date, end_time=end_date)
print(days)

# 轮询获取指数成本股
index_components = {}
end_datetime = datetime.strptime(end_date, "%Y%m%d").replace(tzinfo=DB_TZ)
for ts in days:
    # 毫秒时间戳转 datetime
    dt = datetime.fromtimestamp(ts / 1000, tz=DB_TZ)
    if dt > end_datetime:
        continue

    # 获取成分股（传入日期字符串）
    date_str = dt.strftime("%Y%m%d")
    xt_symbols = xtdata.get_stock_list_in_sector(sector_name, date_str)
    print(f'{sector_name}一共有{len(xt_symbols)}只品种')
    vt_symbols: list = []
    for xt_symbol in xt_symbols:
        vt_symbol = xt_symbol.replace("SH", "SSE").replace("SZ", "SZSE")
        vt_symbols.append(vt_symbol)

    index_components[dt.strftime("%Y-%m-%d")] = vt_symbols

# 保存到数据中心
lab.save_component_data(index_symbol, index_components)

[1704124800000, 1704211200000, 1704297600000, 1704384000000, 1704643200000, 1704729600000, 1704816000000, 1704902400000, 1704988800000, 1705248000000, 1705334400000, 1705420800000, 1705507200000, 1705593600000, 1705852800000, 1705939200000, 1706025600000, 1706112000000, 1706198400000, 1706457600000, 1706544000000, 1706630400000, 1706716800000, 1706803200000, 1707062400000, 1707148800000, 1707235200000, 1707321600000, 1708272000000, 1708358400000, 1708444800000, 1708531200000, 1708617600000, 1708876800000, 1708963200000, 1709049600000, 1709136000000, 1709222400000, 1709481600000, 1709568000000, 1709654400000, 1709740800000, 1709827200000, 1710086400000, 1710172800000, 1710259200000, 1710345600000, 1710432000000, 1710691200000, 1710777600000, 1710864000000, 1710950400000, 1711036800000, 1711296000000, 1711382400000, 1711468800000, 1711555200000, 1711641600000, 1711900800000, 1711987200000, 1712073600000, 1712505600000, 1712592000000, 1712678400000, 1712764800000, 1712851200000, 171311040

In [7]:
# 加载指数成分股代码
component_symbols = lab.load_component_symbols(index_symbol, start_date, end_date)
print(component_symbols)

['688618.SSE', '688201.SSE', '002511.SZSE', '688031.SSE', '603708.SSE', '002975.SZSE', '002958.SZSE', '000068.SZSE', '603049.SSE', '688633.SSE', '002947.SZSE', '000677.SZSE', '688095.SSE', '300530.SZSE', '688096.SSE', '000823.SZSE', '603363.SSE', '603878.SSE', '002467.SZSE', '688308.SSE', '300173.SZSE', '688777.SSE', '003043.SZSE', '000680.SZSE', '301290.SZSE', '603297.SSE', '688317.SSE', '000159.SZSE', '688126.SSE', '600446.SSE', '300402.SZSE', '600884.SSE', '688244.SSE', '000782.SZSE', '300931.SZSE', '000685.SZSE', '601918.SSE', '300796.SZSE', '601326.SSE', '603050.SSE', '601985.SSE', '300119.SZSE', '300376.SZSE', '002094.SZSE', '300589.SZSE', '300968.SZSE', '688275.SSE', '603698.SSE', '600712.SSE', '603697.SSE', '688710.SSE', '002311.SZSE', '000088.SZSE', '688276.SSE', '688305.SSE', '605068.SSE', '300204.SZSE', '300159.SZSE', '603013.SSE', '300964.SZSE', '688607.SSE', '300071.SZSE', '688229.SSE', '605566.SSE', '002983.SZSE', '301101.SZSE', '000615.SZSE', '603262.SSE', '600078.SSE', 

In [8]:
# 转换时间格式
start = datetime.strptime(start_date, "%Y%m%d")
start.replace(tzinfo=DB_TZ)

end = datetime.strptime(end_date, "%Y%m%d")
end.replace(tzinfo=DB_TZ)

# 除了成分股，还要下载指数数据
task_symbols = component_symbols + [index_symbol]

# 遍历下载数据
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    
    for interval in intervals:
        req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval)
        bars = datafeed.query_bar_history(req)

        if bars:
            lab.save_bar_data(bars)
        else:
            logger.error(f"{interval.value} {vt_symbol} 数据下载失败")

 63%|██████▎   | 3280/5201 [23:01<12:28,  2.57it/s]

2026-04-18 03:01:04 d 001312.SZSE 数据下载失败


 71%|███████   | 3672/5201 [25:34<09:51,  2.58it/s]

2026-04-18 03:03:37 d 301513.SZSE 数据下载失败


100%|██████████| 5201/5201 [35:56<00:00,  2.41it/s]


In [9]:
# 添加回测参数配置
for vt_symbol in component_symbols:
    lab.add_contract_setting(
        vt_symbol,
        long_rate=5/10000,
        short_rate=10/10000,
        size=1,
        pricetick=0.0001,
    )